# Gradient Accumulation

**难度:** Easy

实现微批次梯度累积。

梯度累积通过在多个小批次上累积梯度后执行一次优化器步骤，模拟大批次训练。

**签名:** `accumulated_step(model, optimizer, loss_fn, micro_batches) -> float`

**参数:**
- `model` — nn.Module
- `optimizer` — torch 优化器
- `loss_fn` — 损失函数
- `micro_batches` — (x, y) 元组列表

**返回:** 总损失（浮点数）

**约束:**
- 每个微批次损失在反向传播前除以 `n`
- 必须与单次全批次更新数值一致

**提示:**
1. optimizer.zero_grad() 一次
2. 对每个微批次：前向 → loss/n → 反向
3. optimizer.step()
   除以 n 确保累积梯度与单次全批次更新一致。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ INTERVIEW

# ---- 手写梯度累积 ----
# 面试要点：这题本身就不使用高级 API，重点在于理解为什么 loss 要除以 n

def accumulated_step(model, optimizer, loss_fn, micro_batches):
    # 关键：只在最开始清零一次梯度
    optimizer.zero_grad()

    total_loss = 0.0
    n = len(micro_batches)

    for x, y in micro_batches:
        pred = model(x)
        # 面试高频考点：为什么要除以 n？
        # 假设大 batch 的 loss = (1/N) * Σ loss_i
        # 拆成 n 个 micro-batch 后，每个 micro-batch 的 loss = (1/m) * Σ loss_j
        # 如果不除以 n，累积梯度 = Σ d(loss_i)/d(param) = n * d(大batch_loss)/d(param)
        # 除以 n 后：累积梯度 = (1/n) * Σ d(loss_i)/d(param) = d(大batch_loss)/d(param)
        loss = loss_fn(pred, y) / n
        loss.backward()  # PyTorch 默认累加梯度到 .grad
        total_loss += loss.item()

    optimizer.step()  # 更新一次
    return total_loss

In [ ]:
# Demo
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('Accumulated loss:', loss)

In [ ]:
from torch_judge import check
check('gradient_accumulation')

## 📝 核心思路总结

1. **梯度累积 = 小显存模拟大 batch**：在 N 个 micro-batch 上累加梯度，等效于 N 倍大的 batch
2. **loss/n 是关键技巧**：除以 n 保证累积后的梯度与一次性大 batch 计算的梯度数学等价
3. **zero_grad 只调一次**：在累积开始前清零，step 在所有 micro-batch 完成后调用